# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]

# Arboles no requieren escalado, pero si imputacion dentro del pipeline.
tree_preprocessor = ColumnTransformer(
    [("numeric", SimpleImputer(strategy="median"), feature_columns)],
    remainder="drop",
)

# La imputacion y el escalado se ajustan en cada fold de validacion cruzada.
scaled_preprocessor = ColumnTransformer(
    [
        (
            "numeric",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            feature_columns,
        )
    ],
    remainder="drop",
)

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Si se ajusta antes de la validación cruzada, las medias y desviaciones estándar incluirían información de los folds de validación. Al mantenerlo dentro del pipeline, el escalador se ajusta solo con la parte de entrenamiento de cada fold, evitando fuga de información y una estimación optimista del desempeño.

## 3. Modelos candidatos

In [5]:
models = {
    "decision_tree": Pipeline(
        [
            ("preprocessor", tree_preprocessor),
            ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE)),
        ]
    ),
    "logistic_regression": Pipeline(
        [
            ("preprocessor", scaled_preprocessor),
            ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
        ]
    ),
    "knn": Pipeline(
        [
            ("preprocessor", scaled_preprocessor),
            ("classifier", KNeighborsClassifier()),
        ]
    ),
}

## 4. Comparación inicial con validación cruzada

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
initial_rows = []

for name, pipeline in models.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        return_train_score=True,
        n_jobs=1,
    )
    initial_rows.append(
        {
            "model": name,
            "mean_train_accuracy": scores["train_score"].mean(),
            "mean_validation_accuracy": scores["test_score"].mean(),
            "std_validation_accuracy": scores["test_score"].std(ddof=1),
        }
    )

initial_comparison = pd.DataFrame(initial_rows).sort_values(
    "mean_validation_accuracy", ascending=False
).reset_index(drop=True)
print(initial_comparison.to_string(index=False))

              model  mean_train_accuracy  mean_validation_accuracy  std_validation_accuracy
logistic_regression               1.0000                  0.979064                 0.030985
                knn               0.9771                  0.950739                 0.039438
      decision_tree               1.0000                  0.902217                 0.065535


## 5. Optimización de hiperparámetros

In [7]:
param_grids = {
    "decision_tree": {
        "classifier__criterion": ["gini", "entropy"],
        "classifier__max_depth": [None, 3, 5, 8],
        "classifier__min_samples_leaf": [1, 2, 4],
    },
    "logistic_regression": {
        "classifier__C": [0.1, 1.0, 10.0],
        "classifier__solver": ["lbfgs"],
    },
    "knn": {
        "classifier__n_neighbors": [3, 5, 7, 9, 11],
        "classifier__weights": ["uniform", "distance"],
        "classifier__p": [1, 2],
    },
}

searches = {}
tuned_rows = []
for name, pipeline in models.items():
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[name],
        scoring="accuracy",
        cv=cv,
        n_jobs=1,
        return_train_score=True,
        refit=True,
    )
    search.fit(X_train, y_train)
    searches[name] = search
    best_index = search.best_index_
    tuned_rows.append(
        {
            "model": name,
            "best_cv_accuracy": search.best_score_,
            "std_cv_accuracy": search.cv_results_["std_test_score"][best_index],
            "mean_train_accuracy": search.cv_results_["mean_train_score"][best_index],
            "best_params": search.best_params_,
        }
    )

tuned_comparison = pd.DataFrame(tuned_rows).sort_values(
    "best_cv_accuracy", ascending=False
).reset_index(drop=True)
print(tuned_comparison.to_string(index=False))

              model  best_cv_accuracy  std_cv_accuracy  mean_train_accuracy                                                                                         best_params
logistic_regression          0.979064         0.027713             1.000000                                               {'classifier__C': 1.0, 'classifier__solver': 'lbfgs'}
                knn          0.978818         0.017301             0.980624                {'classifier__n_neighbors': 7, 'classifier__p': 1, 'classifier__weights': 'uniform'}
      decision_tree          0.902463         0.076225             0.985934 {'classifier__criterion': 'gini', 'classifier__max_depth': None, 'classifier__min_samples_leaf': 2}


## 6. Selección y persistencia del campeón

In [8]:
champion_name = max(searches, key=lambda name: searches[name].best_score_)
champion = searches[champion_name].best_estimator_
best_search = searches[champion_name]

joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

training_metadata = {
    "champion": champion_name,
    "best_cv_accuracy": float(best_search.best_score_),
    "cv_accuracy_std": float(
        best_search.cv_results_["std_test_score"][best_search.best_index_]
    ),
    "best_params": best_search.best_params_,
    "random_state": RANDOM_STATE,
    "cv_folds": 5,
    "features": feature_columns,
    "target": contract["target"],
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2),
    encoding="utf-8",
)

cv_results = pd.concat(
    [
        pd.DataFrame(search.cv_results_).assign(model=name)
        for name, search in searches.items()
    ],
    ignore_index=True,
)
cv_results.to_csv(REPORTS_DIR / "cv_results.csv", index=False)
print("Modelo campeon:", champion_name)
print("Accuracy CV:", training_metadata["best_cv_accuracy"])

Modelo campeon: logistic_regression
Accuracy CV: 0.9790640394088669


In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

Se eligió `logistic_regression`, con accuracy media de validación cruzada de **0.9791** y desviación estándar de **0.0277** en 5 folds. Superó por un margen muy pequeño a KNN (**0.9788**, desviación estándar **0.0173**); sus resultados son prácticamente equivalentes, así que no se debe interpretar esa diferencia como una ventaja concluyente. El árbol obtuvo menor accuracy (**0.9025**) y mayor variabilidad (**0.0762**). La regresión logística alcanzó accuracy de entrenamiento de 1.0, frente a 0.9791 en validación, lo que sugiere una brecha pequeña que conviene vigilar, aunque su validación es estable. Entre los dos mejores modelos, la regresión logística ofrece una frontera lineal y coeficientes más interpretables que KNN, por lo que es una elección razonable y sencilla. La selección se hizo solo con CV sobre entrenamiento; el test no se utilizó.

## Checklist

- [x] Todo el procesamiento está dentro de pipelines.
- [x] Comparé tres familias de modelos.
- [x] Optimicé hiperparámetros con CV.
- [x] Elegí el campeón sin usar test.
- [x] Guardé el pipeline completo y sus metadatos.